# Kamera autoencoder - harom architektura osszehasonlitasa

Mindharom modell ugyanazt csinalja: egy 160x80-as CARLA kamerakepet tomorit
**128 dimenzios latens vektorra**, majd visszaepiti. A latens megy majd az RL
agent observationjebe.

| modell | felepites | parameter |
|---|---|---|
| `camera_ae` | sima Conv2d stack | 4.7M |
| `resnet_ae` | residual blokkok + multi-res skip | 2.4M |
| `vgg_ae` | VGG-19 elso negy blokkja (ImageNet elotanitott) | 8.7M |

**A notebook felepitese**

1. Modulok betoltese
2. Adatok betoltese
3. Kozos tanito fuggvenyek
4. `camera_ae` tanitasa + gorbek
5. `resnet_ae` tanitasa + gorbek
6. `vgg_ae` tanitasa + gorbek
7. **Osszehasonlitas** azonos kepeken

Minden modell a sajat `camera/<nev>.ckpt` fajljaba mentodik, es minden
tanitas utan felszabadul a VRAM.

## 1. Modulok betoltese

In [ ]:
import gc
import glob
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from camera.camera_ae import CameraAutoEncoder
from camera.resnet_ae import ResNetAE
from camera.vgg_ae import VGGAE

DATA_DIR = "dataset/camera"
LATENT_DIM = 128          # mindharom modellnek ugyanaz
EPOCHS = 30
BATCH = 64

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)

print(DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
def free_vram(*objects):
    """Modellek/tenzorok eldobasa es a GPU cache uritese.

    MIERT KELL: harom modellt tanitunk egymas utan. A PyTorch nem adja
    vissza magatol a memoriat az operacios rendszernek - a cache-ben tartja
    ujrahasznositasra. Ha nem uritjuk, a masodik/harmadik tanitas
    'CUDA out of memory'-val elszall egy 8 GB-os kartyan.
    """
    for o in objects:
        if hasattr(o, "cpu"):
            o.cpu()
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def vram():
    """Aktualis GPU memoriahasznalat GB-ban."""
    if not torch.cuda.is_available():
        return "n/a"
    return (f"{torch.cuda.memory_allocated() / 1e9:.2f} GB foglalt / "
            f"{torch.cuda.memory_reserved() / 1e9:.2f} GB fenntartva")


print("indulaskor:", vram())

## 2. Adatok betoltese

A `dataset/camera/` mappa PNG-i. A kepek 20 Hz-cel, idorendben keszultek -
ezert **keverni kell** oket, kulonben egy batch ugyanannak a par masodpercnek
a majdnem azonos kepeibol allna, es a gradiens torzitana.

In [ ]:
def load_images(data_dir=DATA_DIR):
    """PNG-k -> (N, 3, 80, 160) float32 [0,1]."""
    paths = sorted(glob.glob(os.path.join(data_dir, "*.png")))
    out = np.empty((len(paths), 80, 160, 3), dtype=np.uint8)
    for i, p in enumerate(tqdm(paths, desc="betoltes", unit="kep")):
        out[i] = np.asarray(Image.open(p).convert("RGB"))
    return torch.from_numpy(out).permute(0, 3, 1, 2).float().div_(255.0)


t0 = time.time()
images = load_images()
print(f"{tuple(images.shape)}  {images.numel() * 4 / 1e9:.1f} GB  "
      f"({time.time() - t0:.0f} s)")
print(f"ertekek [{images.min():.2f}, {images.max():.2f}]  atlag {images.mean():.3f}")

In [ ]:
# Keveres, majd train/val vagas.
images = images[torch.randperm(len(images))]

n_val = int(len(images) * 0.15)
x_val, x_train = images[:n_val], images[n_val:]

train_loader = DataLoader(TensorDataset(x_train), batch_size=BATCH, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val), batch_size=BATCH)

print(f"train {len(x_train)}  val {len(x_val)}")
print(f"batch {BATCH}, train batch-ek {len(train_loader)}")

In [ ]:
def show(x, n=8, title=""):
    """Nehany kep megjelenitese."""
    idx = np.random.choice(len(x), n, replace=False)
    fig, axes = plt.subplots(1, n, figsize=(2.1 * n, 2.4))
    for ax, i in zip(axes, idx):
        ax.imshow(x[i].permute(1, 2, 0).numpy())
        ax.axis("off")
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show(x_train, title="Tanito kepek")

## 3. Kozos tanito fuggvenyek

Mindharom modell UGYANAZZAL a fuggvennyel tanul, ugyanazon az adaton es
ugyanazzal a loss-szal (MSE) - kulonben nem a halokat hasonlitanank ossze,
hanem a tanitasi beallitasokat.

In [ ]:
def evaluate(model, loader):
    """Atlagos MSE a teljes loaderen."""
    model.eval()
    total = 0.0
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            total += F.mse_loss(model(xb), xb).item() * len(xb)
    return total / len(loader.dataset)


def train_model(model, label, epochs=EPOCHS, lr=1e-3, patience=5,
                batch=None):
    """Tanitas MSE-vel. A vegen a LEGJOBB val loss-hoz tartozo sulyok maradnak.

    A visszaadott dict tartalmazza a gorbeket es a ckpt utvonalat, de a
    MODELLT NEM - az a tanitas vegen lekerul a GPU-rol es felszabadul.
    Az osszehasonlitashoz a checkpointbol toltjuk vissza.
    """
    # Sajat batch meret, ha a modell nem fer be az alapertelmezettel.
    loader = train_loader
    if batch is not None and batch != BATCH:
        loader = DataLoader(TensorDataset(x_train), batch_size=batch,
                            shuffle=True)

    model = model.to(DEVICE)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)

    hist = {"label": label, "arch": type(model).__name__,
            "train": [], "val": [], "best": float("inf"),
            "params": sum(p.numel() for p in model.parameters()),
            "ckpt": f"camera/{label}.ckpt"}
    best_state, bad = None, 0

    print(f"{label}  ({hist['params'] / 1e6:.1f}M parameter, lr={lr}, "
          f"batch={batch or BATCH})")
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        run = 0.0
        bar = tqdm(loader, desc=f"epoch {ep}/{epochs}", leave=False)
        for (xb,) in bar:
            xb = xb.to(DEVICE)
            loss = F.mse_loss(model(xb), xb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            run += loss.item() * len(xb)
            bar.set_postfix(loss=f"{loss.item():.5f}")

        tr = run / len(loader.dataset)
        va = evaluate(model, val_loader)
        hist["train"].append(tr)
        hist["val"].append(va)

        # Early stopping: ha `patience` epochon at nem javul a val loss,
        # megallunk. A tullanulas ellen ved es idot sporol.
        if va < hist["best"]:
            hist["best"], bad = va, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print(f"  early stop ({patience} epoch javulas nelkul)")
                break

        print(f"  epoch {ep:2d}  train {tr:.5f}  val {va:.5f}"
              f"{'  *' if bad == 0 else ''}")

    model.load_state_dict(best_state)
    hist["epochs_run"] = len(hist["train"])
    hist["time"] = time.time() - t0

    # MENTES sajat nevvel. Az arch is bekerul, hogy visszatoltesnel tudd,
    # melyik architekturarol van szo.
    torch.save({"state_dict": best_state,
                "hparams": dict(model.hparams),
                "arch": hist["arch"],
                "best_val": hist["best"],
                "train": hist["train"], "val": hist["val"]}, hist["ckpt"])

    print(f"  mentve: {hist['ckpt']}  (best val {hist['best']:.5f}, "
          f"{hist['time'] / 60:.1f} perc)")
    return hist, model


def plot_history(hist):
    """Egy modell tanulasi gorbeje."""
    ep = range(1, len(hist["train"]) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(ep, hist["train"], "-o", ms=3, label="train")
    plt.plot(ep, hist["val"], "-s", ms=3, label="val")
    plt.axhline(hist["best"], ls=":", c="gray",
                label=f"best val {hist['best']:.5f}")
    plt.xlabel("epoch")
    plt.ylabel("MSE")
    plt.yscale("log")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.title(f"{hist['label']}  ({hist['params'] / 1e6:.1f}M parameter)")
    plt.tight_layout()
    plt.show()

## 4. `camera_ae`

### Felepites

Negy `Conv2d(kernel=4, stride=2)` blokk, mindegyik felezi a meretet:

```
(3, 80, 160) -> (32, 40, 80) -> (64, 20, 40) -> (128, 10, 20) -> (256, 5, 10)
```

Utana `flatten` (12800) es egyetlen `Linear(12800 -> 128)`.
A dekoder ennek pontos tukorkepe `ConvTranspose2d` retegekkel.

In [ ]:
model = CameraAutoEncoder(latent_dim=LATENT_DIM)
hist_camera_ae, model = train_model(model, label="camera_ae", lr=1e-3)

plot_history(hist_camera_ae)

In [ ]:
# VRAM felszabaditasa a kovetkezo modell elott.
# A sulyok mar a camera/camera_ae.ckpt fajlban vannak, a modell elengedheto.
free_vram(model)
print("camera_ae utan:", vram())

## 5. `resnet_ae`

### Felepites

Residual blokkok (`x + F(x)`) negy szinten, **multi-resolution skip**
kapcsolatokkal: minden szint kimenete kap egy rovidzarat egyenesen a
bottleneck ele, igy a finom reszletek is eljutnak a latensig.

> A `n_ResidualBlock` alapertelmezese **2** (nem az eredeti 8).
> Merve azonos idokeretben a 2 adja a legjobb eredmenyt: kevesebb reteg
> -> gyorsabb lepes -> tobb lepes ugyanannyi ido alatt.

In [ ]:
model = ResNetAE(latent_dim=LATENT_DIM)
hist_resnet_ae, model = train_model(model, label="resnet_ae", lr=1e-3)

plot_history(hist_resnet_ae)

In [ ]:
# VRAM felszabaditasa a kovetkezo modell elott.
# A sulyok mar a camera/resnet_ae.ckpt fajlban vannak, a modell elengedheto.
free_vram(model)
print("resnet_ae utan:", vram())

## 6. `vgg_ae`

### Felepites

A VGG-19 elso negy blokkja (a `relu4_1` retegig) encoderkent,
**ImageNet-en elotanitott sulyokkal**. A dekoder ennek tukorkepe.

> **Ket kulonbseg a masik kettohoz kepest:**
> - `lr=1e-4`, nem `1e-3`. Ez a halo 22 konvolucios reteg normalizalas
>   nelkul; 1e-3 mellett merve elszall (a loss 0.09-rol 0.33-ra ugrik).
> - Az elotanitott sulyok **kotelezoek**: veletlen indulasbol a jel elhal
>   a melyeben, es a halo egyaltalan nem tanul.

In [ ]:
# A VGG a legnagyobb halo (8.7M parameter, 22 konvolucios reteg), ezert
# tobb aktivaciot tarol a backwardhoz. Merve batch=64-gyel ~4 GB kellene,
# ami egy 8 GB-os kartyan mar nem fer be, ha barmi mas is fut rajta
# (pl. egy masik notebook kernel). batch=32 eseten a csucs 2.1 GB.
model = VGGAE(latent_dim=LATENT_DIM)
hist_vgg_ae, model = train_model(model, label="vgg_ae", lr=1e-4, batch=32)

plot_history(hist_vgg_ae)

In [ ]:
# VRAM felszabaditasa a kovetkezo modell elott.
# A sulyok mar a camera/vgg_ae.ckpt fajlban vannak, a modell elengedheto.
free_vram(model)
print("vgg_ae utan:", vram())

## 7. Osszehasonlitas

Mindharom modell **ugyanazokon a validacios kepeken**, amiket egyik sem latott
tanitas kozben. A modelleket a mentett checkpointokbol toltjuk vissza.

In [ ]:
HISTORIES = [hist_camera_ae, hist_resnet_ae, hist_vgg_ae]
ARCHS = {"CameraAutoEncoder": CameraAutoEncoder,
         "ResNetAE": ResNetAE,
         "VGGAE": VGGAE}


def load_model(hist):
    """Modell visszatoltese a checkpointbol."""
    ck = torch.load(hist["ckpt"], map_location="cpu", weights_only=False)
    hp = dict(ck["hparams"])
    # A VGG-nel az elotanitott sulyok letoltese felesleges: a ckpt-bol
    # ugyis felulirjuk oket.
    if ck["arch"] == "VGGAE":
        hp["pretrained"] = False
    m = ARCHS[ck["arch"]](**hp)
    m.load_state_dict(ck["state_dict"])
    return m.eval()


print("Mentett checkpointok:")
for h in HISTORIES:
    size = os.path.getsize(h["ckpt"]) / 1e6
    print(f"  {h['ckpt']:28s} {size:6.1f} MB   best val {h['best']:.5f}")

In [ ]:
# --- Tanulasi gorbek egyutt ---
plt.figure(figsize=(9, 5))
colors = plt.cm.tab10(np.linspace(0, 1, 10))

for k, h in enumerate(HISTORIES):
    ep = range(1, len(h["train"]) + 1)
    plt.plot(ep, h["train"], "-", color=colors[k], label=f"{h['label']} train")
    plt.plot(ep, h["val"], "--", color=colors[k], label=f"{h['label']} val")

plt.xlabel("epoch")
plt.ylabel("MSE")
plt.yscale("log")
plt.grid(alpha=0.3)
plt.legend()
plt.title("Tanulasi gorbek - mindharom modell")
plt.tight_layout()
plt.show()

In [ ]:
# --- Szamszeru osszehasonlitas ---
# A PSNR beszedesebb, mint a nyers MSE: dB-ben meri a jel/zaj aranyt.
# Referencia: 30 dB folott a kulonbseg szabad szemmel alig lathato.

rows = []
for h in sorted(HISTORIES, key=lambda d: d["best"]):
    psnr = 10 * np.log10(1.0 / h["best"])
    px = np.sqrt(h["best"]) * 255          # tipikus pixelhiba 0-255 skalan
    rows.append((h["label"], h["best"], psnr, px, h["params"] / 1e6,
                 h["epochs_run"], h["time"] / 60))

print(f"{'modell':14s} {'val MSE':>9s} {'PSNR':>8s} {'pixelhiba':>10s} "
      f"{'param':>8s} {'epoch':>6s} {'perc':>6s}")
print("-" * 70)
for r in rows:
    print(f"{r[0]:14s} {r[1]:9.5f} {r[2]:7.2f}dB {r[3]:9.1f} "
          f"{r[4]:7.1f}M {r[5]:6d} {r[6]:6.1f}")

print(f"\nLEGJOBB: {rows[0][0]}  (val MSE {rows[0][1]:.5f}, PSNR {rows[0][2]:.2f} dB)")

In [ ]:
# --- Rekonstrukciok egymas alatt, UGYANAZOKON a kepeken ---
n = 8
idx = np.random.choice(len(x_val), n, replace=False)
orig = x_val[idx]

fig, axes = plt.subplots(len(HISTORIES) + 1, n,
                         figsize=(2.1 * n, 2.1 * (len(HISTORIES) + 1)))

for j in range(n):
    axes[0, j].imshow(orig[j].permute(1, 2, 0).numpy())
    axes[0, j].axis("off")
axes[0, 0].set_title("eredeti", loc="left", fontsize=10, weight="bold")

for i, h in enumerate(HISTORIES, start=1):
    m = load_model(h).to(DEVICE)
    with torch.no_grad():
        rec = m(orig.to(DEVICE)).cpu().clamp(0, 1)
    free_vram(m)                      # minden modell utan takaritunk

    for j in range(n):
        axes[i, j].imshow(rec[j].permute(1, 2, 0).numpy())
        axes[i, j].axis("off")
    axes[i, 0].set_title(f"{h['label']}  (val {h['best']:.5f})",
                         loc="left", fontsize=10)

plt.tight_layout()
plt.show()
print("VRAM:", vram())

In [ ]:
# --- Kihasznaljak-e a modellek a latens teret? ---
#
# halott dimenzio : akinek a szorasa ~0, az nem hordoz informaciot
# telitett        : a tanh hataran (+-latent_scale) ulo ertekek - ezek
#                   gradiense majdnem nulla, tehat mar nem tanulnak

n_sample = min(1000, len(x_val))
sidx = np.random.choice(len(x_val), n_sample, replace=False)
sample = x_val[sidx]

fig, axes = plt.subplots(1, len(HISTORIES), figsize=(6 * len(HISTORIES), 3.4),
                         squeeze=False)

for k, h in enumerate(HISTORIES):
    m = load_model(h).to(DEVICE)
    scale = m.hparams.latent_scale
    with torch.no_grad():
        z = m.encode(sample.to(DEVICE)).cpu().numpy()
    free_vram(m)

    std = z.std(axis=0)
    dead = int((std < 0.01 * scale).sum())
    sat = 100 * (np.abs(z) > 0.97 * scale).mean()

    ax = axes[0, k]
    ax.bar(range(len(std)), np.sort(std)[::-1], width=1.0)
    ax.set_title(f"{h['label']}\n{dead} halott dim, {sat:.1f}% telitett",
                 fontsize=10)
    ax.set_xlabel("latens dimenzio (szoras szerint rendezve)")
    ax.set_ylabel("szoras")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("VRAM:", vram())

### Osszegzes

A tablazat `LEGJOBB` sora mondja meg, melyik modell nyert a validacios
MSE alapjan. Amit erdemes melle nezni:

- **PSNR** - 30 dB folott a kulonbseg szabad szemmel alig lathato
- **parameterszam** - kisebb modell gyorsabb az RL futasban is
- **halott dimenziok** - ha sok van, a latens tenyleges merete kisebb,
  mint 128, tehat a modell nem hasznalja ki a rendelkezesre allo helyet
- **telitettseg** - magas ertek azt jelenti, hogy a tanh hataran ul a
  latens; ott a gradiens majdnem nulla

A gyoztes checkpointjat kell beallitani a `config.py` `AE_CKPT_PATH`
erteketkent, es az `LSIZE`-t 128-ra allitani.